# Notebook 05: Contrastive Fine-Tuning

Train two bi-encoder variants using contrastive learning on ToolBench:

- **Variant 2:** In-batch negatives only (random by shuffling)
- **Variant 3:** In-batch + explicit hard negatives (DFSDT + category siblings)

Both use `MultipleNegativesRankingLoss` from sentence-transformers.

**Prerequisite:** Run `01_index_apis.ipynb` first.

In [ ]:
import os, sys, json, random
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))

from data.load_toolbench import load_api_corpus, load_eval_examples
from data.negative_mining import build_api_lookup, build_dfsdt_negatives
from models.embeddings import format_api_string

In [ ]:
corpus = load_api_corpus(TOOLBENCH_DIR / 'toolenv' / 'tools')
lookup = build_api_lookup(corpus)

TRAIN_PATH = TOOLBENCH_DIR / 'toolllama_G123_dfs_train.json'
assert TRAIN_PATH.exists(), f'Training file not found: {TRAIN_PATH}'

train_examples = load_eval_examples(TRAIN_PATH)
with open(TRAIN_PATH) as f:
    raw_train = json.load(f)

print(f'Corpus: {len(corpus)} APIs | Training: {len(train_examples)} examples')

## Configuration

In [ ]:
BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
EPOCHS = 5
BATCH_SIZE = 64
N_HARD_NEGATIVES = 7

## Variant 2: In-Batch Negatives Only

Each training pair is (query, positive API). The loss treats all other positives in the batch as implicit negatives.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

pairs_v2 = []
for ex in train_examples:
    for name in ex['ground_truth_apis']:
        if name in lookup:
            pairs_v2.append(InputExample(texts=[
                ex['user_query'],
                format_api_string(lookup[name]),
            ]))
random.shuffle(pairs_v2)
print(f'Variant 2: {len(pairs_v2)} training pairs')

model_v2 = SentenceTransformer(BASE_MODEL)
loader_v2 = DataLoader(pairs_v2, shuffle=True, batch_size=BATCH_SIZE)
loss_v2 = losses.MultipleNegativesRankingLoss(model_v2)

model_v2.fit(
    train_objectives=[(loader_v2, loss_v2)],
    epochs=EPOCHS,
    warmup_steps=int(0.1 * len(loader_v2) * EPOCHS),
    output_path=str(PROJECT_DIR / 'checkpoints' / 'v2_random'),
    show_progress_bar=True,
)
print('Variant 2 saved to checkpoints/v2_random')

## Variant 3: Explicit Hard Negatives

Each training tuple is (query, positive, neg1, ..., neg7). Negatives come from DFSDT failure paths, padded with category siblings when needed. The loss uses these as explicit negatives on top of in-batch negatives.

In [ ]:
pairs_v3 = []
for ex in train_examples:
    raw_ex = raw_train[ex['raw_idx']]
    for name in ex['ground_truth_apis']:
        if name not in lookup:
            continue
        hard_negs = build_dfsdt_negatives(raw_ex, corpus, ex['ground_truth_apis'], lookup, n=N_HARD_NEGATIVES)
        neg_texts = [format_api_string(n) for n in hard_negs]
        if len(neg_texts) < N_HARD_NEGATIVES:
            continue
        pairs_v3.append(InputExample(texts=[
            ex['user_query'],
            format_api_string(lookup[name]),
        ] + neg_texts))
random.shuffle(pairs_v3)
print(f'Variant 3: {len(pairs_v3)} training pairs')

model_v3 = SentenceTransformer(BASE_MODEL)
loader_v3 = DataLoader(pairs_v3, shuffle=True, batch_size=BATCH_SIZE)
loss_v3 = losses.MultipleNegativesRankingLoss(model_v3)

model_v3.fit(
    train_objectives=[(loader_v3, loss_v3)],
    epochs=EPOCHS,
    warmup_steps=int(0.1 * len(loader_v3) * EPOCHS),
    output_path=str(PROJECT_DIR / 'checkpoints' / 'v3_hard'),
    show_progress_bar=True,
)
print('Variant 3 saved to checkpoints/v3_hard')